In [1]:
!pip install -U bitsandbytes>=0.46.1
!pip install -q transformers peft accelerate bitsandbytes
!pip install fastapi uvicorn pyngrok nest_asyncio
!pip install langchain langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.0 MB/s eta 0:00:00


In [2]:
import re
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from peft import PeftModel
from langchain.messages import AIMessage
from langchain_groq import ChatGroq
from pydantic import SecretStr
from dotenv import load_dotenv
import torch
from google.colab import userdata

In [3]:
 load_dotenv()

False

In [4]:

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

HF_TOKEN = userdata.get('HF_TOKEN')
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "bigcode/starcoder2-3b"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(model_name, token=HF_TOKEN, quantization_config=bnb_config).to(device)

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [7]:
!unzip -o /content/starcoder2-python-java-custom-lora.zip -d /content/starcoder2-python-java-custom-lora/

Archive:  /content/starcoder2-python-java-custom-lora.zip
  inflating: /content/starcoder2-python-java-custom-lora/adapter_config.json  
  inflating: /content/starcoder2-python-java-custom-lora/tokenizer.json  
  inflating: /content/starcoder2-python-java-custom-lora/adapter_model.safetensors  
  inflating: /content/starcoder2-python-java-custom-lora/training_args.bin  
  inflating: /content/starcoder2-python-java-custom-lora/tokenizer_config.json  
  inflating: /content/starcoder2-python-java-custom-lora/README.md  


In [8]:
# Load the LoRA adapters from the checkpoint
model_to_test = PeftModel.from_pretrained(model, "/content/starcoder2-python-java-custom-lora")

# Set the model to evaluation mode and move to device
model_to_test = model_to_test.eval().to(device)

print("Model loaded successfully for testing.")

Model loaded successfully for testing.


In [9]:

app = FastAPI()

class Request(BaseModel):
    query: str
    task: str
    source_language: str
    target_language: str
    source_code: str
    output_code: str

@app.post("/generate")
def generate(req: Request) -> str:

    prompt_text = f"""
        Generate the code for the following instruction.
        ### Instruction:
        {req.query}

        ### Response
        """

    result = invoke_model(prompt_text)
    return result

@app.post("/translate")
def translate(req: Request) -> str:

    prompt_text = f"""
    {req.source_language} to {req.target_language}

    ### Instruction
    {req.source_code}

    ### Response
    """

    result = invoke_model(prompt_text)
    return result

@app.post("/repair")
def repair(req: Request) -> str:

    prompt_text = f"""
    ### Instruction
    The following code has compilation errors. Please fix and generate the corrected code.
    problem statement for the code: {req.query}

    ### Input
    {req.output_code}
    ### Response
    """
    result = invoke_model(prompt_text)
    return result

def invoke_model(prompt_text: str) -> str:

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model_to_test.device)

    outputs = model_to_test.generate(
        **inputs,
        max_new_tokens=768,
         do_sample=True,
        temperature=0.2,
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return result.split("### Response")[1].strip()

In [16]:
# app = FastAPI()

# @app.get("/")
# def get():
#     return "Hello World"


In [10]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from threading import Thread
from google.colab import userdata

nest_asyncio.apply()

# Get NGROK_AUTH_TOKEN from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

Thread(
    target=lambda: uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    ),
    daemon=True
).start()

public_url = ngrok.connect(8000)

print("Public URL:", public_url)
print("Swagger :", f"{public_url}/docs")

INFO:     Started server process [1926]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://eccentric-blanching-childlike.ngrok-free.dev" -> "http://localhost:8000"
Swagger : NgrokTunnel: "https://eccentric-blanching-childlike.ngrok-free.dev" -> "http://localhost:8000"/docs
